<a href="https://colab.research.google.com/github/vlasovdima/ai-trainer-portfolio/blob/main/evaluation_and_ranking_of_results.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Оценка и ранжирование ответов (RLHF Benchmark)

Промпт пользователя:

"Я разворачиваю Python-приложение на виртуальной машине Ubuntu через Oracle VM VirtualBox (хост — Windows). Напиши пошаговую инструкцию, как правильно настроить проброс портов (NAT), написать systemd-unit для автоматического перезапуска скрипта при падении и настроить логирование через journalctl."

## Вариант ответа А
1. Проброс портов в VirtualBox:
Зайди в настройки виртуальной машины -> Сеть -> Тип подключения: NAT. Нажми «Проброс портов». Добавь новое правило: укажи Порт хоста (например, 8080) и Порт гостя (8080). IP-адреса можно оставить пустыми.

2. Настройка systemd-unit:
Создай файл /etc/systemd/system/myapp.service:

In [ ]:
[Unit]
Description=My Python App

[Service]
ExecStart=python app.py
Restart=yes
User=root

[Install]
WantedBy=multi-user.target

Выполни команду systemctl enable myapp.service и systemctl start myapp.service.

3. Логирование:
Systemd автоматически пишет логи. Чтобы их посмотреть, просто введи команду journalctl в терминале, и ты увидишь вывод твоего скрипта.

## Вариант ответа Б
1. Настройка проброса портов (NAT) в VirtualBox:
В настройках ВМ перейди в Сеть -> Адаптер 1 (NAT) -> Дополнительно -> Проброс портов.
Создай правило:

IP хоста: 127.0.0.1 (важно для безопасности, чтобы порт был доступен только с твоего Windows-хоста, а не из внешней сети).

Порт хоста: 8080 (или любой свободный).

IP гостя: можно оставить пустым или указать IP машины в NAT (обычно 10.0.2.15).

Порт гостя: порт, который слушает твое приложение внутри Ubuntu.

2. Создание systemd-unit:
Создай файл конфигурации: sudo nano /etc/systemd/system/python-app.service.

In [ ]:
[Unit]
Description=Python Application Service
After=network.target

[Service]
User=ubuntu
WorkingDirectory=/home/ubuntu/myproject
ExecStart=/usr/bin/python3 /home/ubuntu/myproject/app.py
Restart=on-failure
RestartSec=5
StandardOutput=journal
StandardError=journal

[Install]
WantedBy=multi-user.target

Важно: Замени ubuntu на твоего пользователя и укажи абсолютные пути к Python и скрипту.
Примени изменения: sudo systemctl daemon-reload, затем sudo systemctl enable --now python-app.service.

3. Логирование через journalctl:
Так как мы указали вывод в journal, логи можно читать целенаправленно для этого сервиса:

* Чтение логов сервиса: journalctl -u python-app.service

* Чтение в реальном времени (как tail -f): journalctl -u python-app.service -f

* Логи со времени последней загрузки: journalctl -u python-app.service -b

# Оценка
Победитель: Вариант Б

Обоснование выбора (Критерии оценки):

Безопасность и Best Practices (Критический фактор):

В Варианте А предложено запускать сервис от имени пользователя root (User=root). Это грубейшее нарушение безопасности для web-приложений. Вариант Б корректно предлагает использовать обычного пользователя.

В Варианте А IP-адрес хоста при пробросе портов остается пустым (0.0.0.0 по умолчанию). Это делает порт открытым для всей локальной сети Windows-хоста. Вариант Б рекомендует привязку к 127.0.0.1, что является правильным